# FlashAttention-3 Benchmark — 1k to 64k Sequence Lengths

**Repo:** [ai-attention-throughput-optimizer](https://github.com/TylrDn/ai-attention-throughput-optimizer)  
**Phase:** 3 — Notebook Pipelines

Benchmarks attention implementations across sequence lengths from 1 k to 64 k tokens,
measuring throughput (tokens/s), peak memory, and latency.

### References
- FlashAttention-2: Dao et al. (2023) https://arxiv.org/abs/2307.08691
- FlashAttention-3: Shah et al. (2024) https://arxiv.org/abs/2407.08608
- Linear Attention: Katharopoulos et al. (2020) https://arxiv.org/abs/2006.16236

In [ ]:
from __future__ import annotations

import logging
import os
import time
from dataclasses import dataclass, field
from typing import Callable

import torch
import wandb

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WANDB_DISABLED = os.getenv("WANDB_DISABLED", "false").lower() == "true"
logger.info("Using device: %s", DEVICE)

In [ ]:
@dataclass
class BenchmarkConfig:
    """Hyperparameters for the attention benchmark."""

    batch_size: int = 2
    num_heads: int = 16
    head_dim: int = 64
    seq_lengths: list[int] = field(
        default_factory=lambda: [1024, 2048, 4096, 8192, 16384, 32768, 65536]
    )
    warmup_steps: int = 3
    bench_steps: int = 10
    dtype: torch.dtype = torch.bfloat16


cfg = BenchmarkConfig()

if not WANDB_DISABLED:
    run = wandb.init(
        project="ai-attention-throughput-optimizer",
        config={
            "batch_size": cfg.batch_size,
            "num_heads": cfg.num_heads,
            "head_dim": cfg.head_dim,
            "seq_lengths": cfg.seq_lengths,
            "dtype": str(cfg.dtype),
        },
    )
logger.info("BenchmarkConfig: %s", cfg)

In [ ]:
import torch.nn.functional as F


def vanilla_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Standard scaled dot-product attention (O(n²) memory).

    Vaswani et al. (2017) https://arxiv.org/abs/1706.03762
    """
    scale = q.size(-1) ** -0.5
    attn = torch.softmax(q @ k.transpose(-2, -1) * scale, dim=-1)
    return attn @ v


def sdpa_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """PyTorch 2.x scaled_dot_product_attention — auto-selects FlashAttention kernel.

    FlashAttention-2: Dao et al. (2023) https://arxiv.org/abs/2307.08691
    """
    # F.scaled_dot_product_attention dispatches to FlashAttention on CUDA with
    # suitable dtypes; falls back to math attention on CPU.
    return F.scaled_dot_product_attention(q, k, v, is_causal=True)


ATTENTION_VARIANTS: dict[str, Callable] = {
    "vanilla": vanilla_attention,
    "sdpa_flash": sdpa_attention,
}

logger.info("Attention variants registered: %s", list(ATTENTION_VARIANTS))

In [ ]:
import gc


def benchmark_attention(
    fn: Callable,
    seq_len: int,
    cfg: BenchmarkConfig,
    device: torch.device = DEVICE,
) -> dict[str, float]:
    """Measure throughput and peak memory for a single (fn, seq_len) combination."""
    B, H, S, D = cfg.batch_size, cfg.num_heads, seq_len, cfg.head_dim
    shape = (B, H, S, D)

    q = torch.randn(shape, dtype=cfg.dtype, device=device)
    k = torch.randn(shape, dtype=cfg.dtype, device=device)
    v = torch.randn(shape, dtype=cfg.dtype, device=device)

    # Warmup
    for _ in range(cfg.warmup_steps):
        _ = fn(q, k, v)
    if device.type == "cuda":
        torch.cuda.synchronize()

    # Reset peak memory
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    t0 = time.perf_counter()
    for _ in range(cfg.bench_steps):
        _ = fn(q, k, v)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    tokens_per_step = B * S
    throughput = tokens_per_step * cfg.bench_steps / elapsed
    peak_mem_gb = (
        torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else float("nan")
    )

    del q, k, v
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "seq_len": seq_len,
        "throughput_tokens_per_s": throughput,
        "latency_ms": elapsed / cfg.bench_steps * 1000,
        "peak_mem_gb": peak_mem_gb,
    }


logger.info("Benchmark harness defined")

In [ ]:
import pandas as pd

results: list[dict] = []

for variant_name, attn_fn in ATTENTION_VARIANTS.items():
    for seq_len in cfg.seq_lengths:
        # Skip large seq lengths for vanilla to avoid OOM
        if variant_name == "vanilla" and seq_len > 8192:
            logger.warning("Skipping vanilla attention at seq_len=%d (OOM risk)", seq_len)
            continue
        try:
            metrics = benchmark_attention(attn_fn, seq_len, cfg)
            metrics["variant"] = variant_name
            results.append(metrics)
            logger.info(
                "%s @ %dk: %.1f k tokens/s  |  %.1f ms  |  %.2f GB",
                variant_name,
                seq_len // 1024,
                metrics["throughput_tokens_per_s"] / 1000,
                metrics["latency_ms"],
                metrics["peak_mem_gb"],
            )
            if not WANDB_DISABLED:
                wandb.log({f"{variant_name}/{k}": v for k, v in metrics.items() if k != "variant"})
        except RuntimeError as exc:
            logger.error("OOM at %s seq_len=%d: %s", variant_name, seq_len, exc)

df = pd.DataFrame(results)
df.to_csv("attention_benchmark_results.csv", index=False)
logger.info("Results saved to attention_benchmark_results.csv")
df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for variant in df["variant"].unique():
    sub = df[df["variant"] == variant]
    axes[0].plot(sub["seq_len"], sub["throughput_tokens_per_s"] / 1e3, marker="o", label=variant)
    axes[1].plot(sub["seq_len"], sub["latency_ms"], marker="o", label=variant)

for ax, ylabel, title in zip(
    axes,
    ["Throughput (k tokens/s)", "Latency (ms)"],
    ["Throughput vs Sequence Length", "Latency vs Sequence Length"],
):
    ax.set_xlabel("Sequence Length")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.set_xscale("log", base=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("attention_benchmark.png", dpi=150)
plt.show()

if not WANDB_DISABLED:
    wandb.log({"benchmark_plot": wandb.Image("attention_benchmark.png")})
    artifact = wandb.Artifact("attention-benchmark", type="results")
    artifact.add_file("attention_benchmark_results.csv")
    artifact.add_file("attention_benchmark.png")
    wandb.log_artifact(artifact)